# 02 - Correlation structure

Visualizes the CSV's verified 3x-temporal-transform redundancy (`prev_week_X`/`X_avg_all`/
`X_avg3` triplets) and offense/defense-mirror pairs that `feature_selection/correlation_pruning.py`
Stage 1 targets. See `docs/feature_selection_methodology.md` for the actual observed pruning
yield on the real data.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd

from cfb_spread_model.data import build_temporal_triplet_groups
from cfb_spread_model.utils.paths import DATA_PROCESSED_DIR

df = pd.read_parquet(DATA_PROCESSED_DIR / "modeling_dataset.parquet")
groups = build_temporal_triplet_groups(list(df.columns))
len(groups)

In [ ]:
# Pick one metric and look at the within-triplet correlation
(side, base), cols = next(iter(groups.items()))
print(side, base, cols)
df[list(cols.values())].corr()

In [ ]:
# Distribution of within-triplet max correlation, across all 336 metric x side groups
import numpy as np

max_corrs = []
for (side, base), transform_map in groups.items():
    cols = list(transform_map.values())
    if len(cols) < 2:
        continue
    corr = df[cols].corr().to_numpy()
    np.fill_diagonal(corr, np.nan)
    max_corrs.append(np.nanmax(np.abs(corr)))

pd.Series(max_corrs).describe()